In [10]:
import sys
!{sys.executable} -m pip install openai

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.9 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [openai]11/12 [openai]c]


In [40]:
import json
import os
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import pandas as pd
from openai import OpenAI

In [41]:
MODEL_NAME = "gpt-4.1"
TEMPERATURES = [0.2, 0.7]
MAX_RETRIES = 3
SLEEP_BETWEEN_CALLS = 1.0

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "sensitivity_outputs"
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw_outputs"

OUTPUT_DIR.mkdir(exist_ok=True)
RAW_OUTPUT_DIR.mkdir(exist_ok=True)

print("Base directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("Raw output directory:", RAW_OUTPUT_DIR)

Base directory: /Users/mollyqiao
Output directory: /Users/mollyqiao/sensitivity_outputs
Raw output directory: /Users/mollyqiao/sensitivity_outputs/raw_outputs


In [42]:
inventory = [
    {"product": "Scented Candle Set", "price": 18, "stock": 120, "category": "Home Decor"},
    {"product": "Essential Oil Diffuser", "price": 35, "stock": 60, "category": "Wellness"},
    {"product": "Lavender Essential Oil", "price": 15, "stock": 200, "category": "Wellness"},
    {"product": "Aromatherapy Bath Salts", "price": 14, "stock": 90, "category": "Wellness"},
    {"product": "Ceramic Coffee Mug", "price": 10, "stock": 150, "category": "Kitchen"},
    {"product": "Gourmet Coffee Sampler", "price": 22, "stock": 80, "category": "Food"},
    {"product": "Wooden Picture Frame", "price": 12, "stock": 85, "category": "Home Decor"},
    {"product": "Linen Throw Blanket", "price": 45, "stock": 70, "category": "Home Decor"},
]

inventory_pretty = json.dumps(inventory, indent=2)
print(inventory_pretty)

[
  {
    "product": "Scented Candle Set",
    "price": 18,
    "stock": 120,
    "category": "Home Decor"
  },
  {
    "product": "Essential Oil Diffuser",
    "price": 35,
    "stock": 60,
    "category": "Wellness"
  },
  {
    "product": "Lavender Essential Oil",
    "price": 15,
    "stock": 200,
    "category": "Wellness"
  },
  {
    "product": "Aromatherapy Bath Salts",
    "price": 14,
    "stock": 90,
    "category": "Wellness"
  },
  {
    "product": "Ceramic Coffee Mug",
    "price": 10,
    "stock": 150,
    "category": "Kitchen"
  },
  {
    "product": "Gourmet Coffee Sampler",
    "price": 22,
    "stock": 80,
    "category": "Food"
  },
  {
    "product": "Wooden Picture Frame",
    "price": 12,
    "stock": 85,
    "category": "Home Decor"
  },
  {
    "product": "Linen Throw Blanket",
    "price": 45,
    "stock": 70,
    "category": "Home Decor"
  }
]


In [43]:
system_prompt_v1 = """
You are an AI-powered product recommendation engine for small retail businesses.

Your task is to analyze a store's inventory and generate product bundle ideas.

Given the inventory data, you should:

1. Recommend 2 complementary product bundles.
2. Suggest a bundle price that provides at least 10% savings compared to the total individual prices.
3. Write a short marketing tagline for each bundle.

Keep the output clear and readable.
""".strip()

system_prompt_v2 = """
You are an AI assistant helping small retail businesses create product bundles.

Given a list of inventory items, generate 2 realistic product bundles.

Follow these steps:

1. Identify products that are complementary in function or theme.
2. Consider stock levels and prioritize items with higher inventory.
3. Calculate the total individual price of the bundle.
4. Apply a bundle discount that provides at least 10% customer savings.
5. Identify the likely target customer for the bundle.
6. Create a short marketing tagline suitable for promotions.

Present the results in a clear and structured format.
""".strip()

system_prompt_v3 = """
You are a reliable product bundling and marketing assistant for small businesses.

Your goal is to generate practical and commercially sensible product bundles based on the store's inventory.

For each bundle:

- Choose products that logically work well together for a specific customer experience.
- Consider stock levels and prioritize items that businesses may want to move.
- Calculate the original total price of the items.
- Suggest a discounted bundle price that gives customers at least 10% savings.
- Ensure the recommendation remains financially reasonable for the business.
- Write a concise bundle name and a short promotional tagline.

Be consistent, structured, and business-focused in your recommendations.
""".strip()

SYSTEM_PROMPTS = {
    "V1": system_prompt_v1,
    "V2": system_prompt_v2,
    "V3": system_prompt_v3,
}

print(SYSTEM_PROMPTS.keys())

dict_keys(['V1', 'V2', 'V3'])


In [44]:
user_input_A = f"""
Using the following inventory, recommend 2 product bundles.

For each bundle include:
- bundle name
- products included
- original total price
- discounted bundle price
- short marketing tagline

Inventory:
{inventory_pretty}
""".strip()

user_input_B = f"""
Create two complementary retail bundles using this inventory.

For each bundle:
- show which products are included
- calculate the bundle price with savings
- write a short promotional message

Inventory:
{inventory_pretty}
""".strip()

user_input_C = f"""
Suggest two realistic product sets that a small store could sell as bundles.

Each bundle should:
- combine related products
- offer at least 10% customer savings
- include a short marketing tagline

Inventory:
{inventory_pretty}
""".strip()

USER_INPUTS = {
    "A": user_input_A,
    "B": user_input_B,
    "C": user_input_C,
}

print(USER_INPUTS.keys())

dict_keys(['A', 'B', 'C'])


In [45]:
@dataclass
class ExperimentRun:
    run_id: str
    prompt_version: str
    input_version: str
    temperature: float
    system_prompt: str
    user_prompt: str
    output_file: str
    raw_output: str

In [24]:
def get_client() -> OpenAI:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Please export your API key before running this script."
        )
    return OpenAI(api_key=api_key)

In [46]:
def sanitize_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)

def build_run_id(prompt_version: str, input_version: str, temperature: float) -> str:
    return f"{prompt_version}_{input_version}_temp{str(temperature).replace('.', '_')}"

def save_text(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")

In [47]:
def call_model(client, system_prompt: str, user_prompt: str, temperature: float, model_name: str = MODEL_NAME) -> str:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=model_name,
                temperature=temperature,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return response.output_text

        except Exception as exc:
            last_error = exc
            print(f"[Retry {attempt}/{MAX_RETRIES}] API call failed: {exc}")
            time.sleep(2 * attempt)

    raise RuntimeError(f"Model call failed after {MAX_RETRIES} attempts.") from last_error

In [34]:
def run_all_experiments() -> List[ExperimentRun]:
    client = get_client()
    all_runs: List[ExperimentRun] = []

    print("Starting prompt sensitivity experiment...")
    print(f"Model: {MODEL_NAME}")
    print(f"Temperatures: {TEMPERATURES}")
    print(f"Prompt versions: {list(SYSTEM_PROMPTS.keys())}")
    print(f"Input versions: {list(USER_INPUTS.keys())}")
    print("-" * 60)

    total = len(SYSTEM_PROMPTS) * len(USER_INPUTS) * len(TEMPERATURES)
    counter = 0

    for prompt_version, system_prompt in SYSTEM_PROMPTS.items():
        for input_version, user_prompt in USER_INPUTS.items():
            for temperature in TEMPERATURES:
                counter += 1
                run_id = build_run_id(prompt_version, input_version, temperature)
                output_filename = sanitize_filename(f"{run_id}.txt")
                output_path = RAW_OUTPUT_DIR / output_filename

                print(f"[{counter}/{total}] Running {run_id} ...")

                raw_output = call_model(
                    client=client,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    temperature=temperature,
                )

                save_text(output_path, raw_output)

                run = ExperimentRun(
                    run_id=run_id,
                    prompt_version=prompt_version,
                    input_version=input_version,
                    temperature=temperature,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    output_file=str(output_path.relative_to(BASE_DIR)),
                    raw_output=raw_output,
                )
                all_runs.append(run)

                time.sleep(SLEEP_BETWEEN_CALLS)

    return all_runs

In [48]:
test_output = call_model(
    client=client,
    system_prompt=SYSTEM_PROMPTS["V1"],
    user_prompt=USER_INPUTS["A"],
    temperature=0.2
)

print(test_output)

**Bundle 1: Relax & Unwind Set**  
- **Products Included:**  
  - Scented Candle Set ($18)  
  - Aromatherapy Bath Salts ($14)  
  - Lavender Essential Oil ($15)  
- **Original Total Price:** $47  
- **Discounted Bundle Price:** $42  
- **Tagline:**  
  *"Turn your home into a spa—relax, soak, and breathe easy with our Relax & Unwind Set!"*

---

**Bundle 2: Cozy Coffee Morning Kit**  
- **Products Included:**  
  - Ceramic Coffee Mug ($10)  
  - Gourmet Coffee Sampler ($22)  
  - Linen Throw Blanket ($45)  
- **Original Total Price:** $77  
- **Discounted Bundle Price:** $69  
- **Tagline:**  
  *"Start your day wrapped in comfort—sip, savor, and snuggle with the Cozy Coffee Morning Kit!"*


In [50]:
def run_all_experiments():
    all_runs = []

    total = len(SYSTEM_PROMPTS) * len(USER_INPUTS) * len(TEMPERATURES)
    counter = 0

    print("Starting prompt sensitivity experiment...")
    print(f"Model: {MODEL_NAME}")
    print(f"Total runs: {total}")
    print("-" * 60)

    for prompt_version, system_prompt in SYSTEM_PROMPTS.items():
        for input_version, user_prompt in USER_INPUTS.items():
            for temperature in TEMPERATURES:
                counter += 1
                run_id = build_run_id(prompt_version, input_version, temperature)
                output_filename = sanitize_filename(f"{run_id}.txt")
                output_path = RAW_OUTPUT_DIR / output_filename

                print(f"[{counter}/{total}] Running {run_id} ...")

                raw_output = call_model(
                    client=client,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    temperature=temperature,
                )

                save_text(output_path, raw_output)

                run = ExperimentRun(
                    run_id=run_id,
                    prompt_version=prompt_version,
                    input_version=input_version,
                    temperature=temperature,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    output_file=str(output_path.relative_to(BASE_DIR)),
                    raw_output=raw_output,
                )

                all_runs.append(run)
                time.sleep(SLEEP_BETWEEN_CALLS)

    return all_runs

In [51]:
all_runs = run_all_experiments()
print(f"Finished {len(all_runs)} runs.")

Starting prompt sensitivity experiment...
Model: gpt-4.1
Total runs: 18
------------------------------------------------------------
[1/18] Running V1_A_temp0_2 ...
[2/18] Running V1_A_temp0_7 ...
[3/18] Running V1_B_temp0_2 ...
[4/18] Running V1_B_temp0_7 ...
[5/18] Running V1_C_temp0_2 ...
[6/18] Running V1_C_temp0_7 ...
[7/18] Running V2_A_temp0_2 ...
[8/18] Running V2_A_temp0_7 ...
[9/18] Running V2_B_temp0_2 ...
[10/18] Running V2_B_temp0_7 ...
[11/18] Running V2_C_temp0_2 ...
[12/18] Running V2_C_temp0_7 ...
[13/18] Running V3_A_temp0_2 ...
[14/18] Running V3_A_temp0_7 ...
[15/18] Running V3_B_temp0_2 ...
[16/18] Running V3_B_temp0_7 ...
[17/18] Running V3_C_temp0_2 ...
[18/18] Running V3_C_temp0_7 ...
Finished 18 runs.


In [52]:
from pathlib import Path
import pandas as pd
from dataclasses import asdict

results_df = pd.DataFrame([asdict(run) for run in all_runs])

desktop_path = Path.home() / "Desktop"

results_csv_path = desktop_path / "sensitivity_results.csv"

results_df.to_csv(results_csv_path, index=False, encoding="utf-8")

print("Saved results CSV to:", results_csv_path)

results_df.head()

Saved results CSV to: /Users/mollyqiao/Desktop/sensitivity_results.csv


,run_id,prompt_version,input_version,temperature,system_prompt,user_prompt,output_file,raw_output
0,V1_A_temp0_2,V1,A,0.2,You are an AI-powered product recommendation e...,"Using the following inventory, recommend 2 pro...",sensitivity_outputs/raw_outputs/V1_A_temp0_2.txt,**Bundle 1: Relax & Unwind Set** \n- **Produc...
1,V1_A_temp0_7,V1,A,0.7,You are an AI-powered product recommendation e...,"Using the following inventory, recommend 2 pro...",sensitivity_outputs/raw_outputs/V1_A_temp0_7.txt,### Bundle 1: Serenity Spa Set\n**Products Inc...
2,V1_B_temp0_2,V1,B,0.2,You are an AI-powered product recommendation e...,Create two complementary retail bundles using ...,sensitivity_outputs/raw_outputs/V1_B_temp0_2.txt,**Bundle 1: Relax & Unwind Set**\n\n**Included...
3,V1_B_temp0_7,V1,B,0.7,You are an AI-powered product recommendation e...,Create two complementary retail bundles using ...,sensitivity_outputs/raw_outputs/V1_B_temp0_7.txt,**Bundle 1: Relax & Unwind Set**\n\n**Includes...
4,V1_C_temp0_2,V1,C,0.2,You are an AI-powered product recommendation e...,Suggest two realistic product sets that a smal...,sensitivity_outputs/raw_outputs/V1_C_temp0_2.txt,**Bundle 1: Relax & Unwind Spa Set** \n- **In...


In [53]:
manifest = {
    "model_name": MODEL_NAME,
    "temperatures": TEMPERATURES,
    "num_runs": len(all_runs),
    "prompt_versions": list(SYSTEM_PROMPTS.keys()),
    "input_versions": list(USER_INPUTS.keys()),
    "inventory": inventory,
    "outputs_folder": str(RAW_OUTPUT_DIR.relative_to(BASE_DIR)),
}

manifest_path = OUTPUT_DIR / "experiment_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest to:", manifest_path)
manifest

Saved manifest to: /Users/mollyqiao/sensitivity_outputs/experiment_manifest.json


{'model_name': 'gpt-4.1',
 'temperatures': [0.2, 0.7],
 'num_runs': 18,
 'prompt_versions': ['V1', 'V2', 'V3'],
 'input_versions': ['A', 'B', 'C'],
 'inventory': [{'product': 'Scented Candle Set',
   'price': 18,
   'stock': 120,
   'category': 'Home Decor'},
  {'product': 'Essential Oil Diffuser',
   'price': 35,
   'stock': 60,
   'category': 'Wellness'},
  {'product': 'Lavender Essential Oil',
   'price': 15,
   'stock': 200,
   'category': 'Wellness'},
  {'product': 'Aromatherapy Bath Salts',
   'price': 14,
   'stock': 90,
   'category': 'Wellness'},
  {'product': 'Ceramic Coffee Mug',
   'price': 10,
   'stock': 150,
   'category': 'Kitchen'},
  {'product': 'Gourmet Coffee Sampler',
   'price': 22,
   'stock': 80,
   'category': 'Food'},
  {'product': 'Wooden Picture Frame',
   'price': 12,
   'stock': 85,
   'category': 'Home Decor'},
  {'product': 'Linen Throw Blanket',
   'price': 45,
   'stock': 70,
   'category': 'Home Decor'}],
 'outputs_folder': 'sensitivity_outputs/raw_ou

## Prompt Sensitivity Experiment Overview

To evaluate prompt sensitivity, we tested how the system behaved under different prompt formulations, user input paraphrases, and temperature settings.

The experiment used:

- **3 system prompt versions**: V1 (baseline), V2 (structured reasoning), and V3 (business-focused prompt)
- **3 paraphrased user inputs**: A, B, and C
- **2 temperature settings**: 0.2 and 0.7

This resulted in **18 total runs (3 × 3 × 2)**.

The inventory dataset was kept fixed across all runs to ensure that any changes in output were caused by prompt variations rather than changes in input data.

Each run generated product bundles including:

- bundle name
- product list
- original total price
- discounted bundle price
- marketing tagline

The outputs were saved in `sensitivity_results.csv` for analysis.

## General Observations

Across the 18 runs, the model successfully generated bundle recommendations for every prompt configuration.

Most outputs contained:

- two product bundles
- clear product combinations
- bundle pricing with savings
- short marketing messages

Two common bundle themes appeared repeatedly across runs:

1. **Wellness / Relaxation bundles**  
   Typically combining items such as Essential Oil Diffuser, Lavender Essential Oil, and Bath Salts.

2. **Cozy Coffee / Comfort bundles**  
   Usually including Ceramic Coffee Mug, Gourmet Coffee Sampler, and sometimes Throw Blanket or Candle.

This consistency suggests that the model reliably identified complementary product relationships within the inventory.

## Effect of System Prompt Variants

The three system prompts produced slightly different behavior.

**V1 (Baseline Prompt)**  
The baseline prompt generated reasonable bundles but sometimes produced less structured outputs.  
Pricing logic and bundle explanations were present but less consistent.

**V2 (Structured Prompt)**  
The structured prompt improved reasoning clarity by guiding the model through bundle logic, pricing, and customer targeting.  
Outputs were generally more organized and easier to interpret.

**V3 (Business-Focused Prompt)**  
The business-focused prompt produced the most practical and production-ready outputs.  
Bundles were well structured, pricing was clearly presented, and marketing language was more aligned with real retail use cases.

Overall, **V3 produced the most balanced and consistent results across runs.**

## Effect of Temperature

Two temperature values were tested:

- **0.2 (low temperature)**  
- **0.7 (higher temperature)**

At **temperature 0.2**, the outputs were more stable and consistent.  
Bundle structures and pricing logic were usually clearer and followed the constraints more reliably.

At **temperature 0.7**, the outputs showed slightly more variation in wording and bundle composition.  
Marketing language sometimes became more creative, but the structure occasionally became less consistent.

For this task, which emphasizes business logic and consistency, **temperature 0.2 generally produced more reliable results.**

## Effect of Input Paraphrasing

The experiment also tested three paraphrased user instructions (A, B, and C).

Although the wording of the instructions differed slightly, the overall recommendations remained similar across runs.  
The model consistently generated comparable bundle themes and pricing structures.

This indicates that the system prompt configuration is **reasonably robust to small variations in user input phrasing**.

## Sensitivity Experiment Conclusion

The sensitivity experiment demonstrates that prompt structure influences both the clarity and reliability of the model’s outputs.

Key observations include:

- The model consistently identified complementary product bundles across runs.
- Structured prompts improved reasoning transparency and output organization.
- The business-focused prompt (V3) produced the most practical and stable results.
- Lower temperature (0.2) improved output consistency.
- Minor changes in user input phrasing did not significantly affect bundle recommendations.

Based on these findings, the **V3 system prompt with temperature 0.2** was selected as the final configuration for the recommendation engine.

## Step 2: Curate, Prepare, and Synthesize Data

In this step, we build a structured dataset for the product bundle recommendation system.

The dataset includes three types of cases:

- **Typical cases**: normal retail situations with naturally complementary products  
- **Edge cases**: constrained or unusual inventory situations  
- **Adversarial cases**: difficult cases with weak product coherence or challenging bundle logic  

This dataset will support later evaluation, fine-tuning, and possible RAG grounding.

In [56]:
import json
import random
from pathlib import Path

import pandas as pd

random.seed(42)

output_dir = Path("step2_dataset_outputs")
output_dir.mkdir(exist_ok=True)

print("Output directory:", output_dir.resolve())

Output directory: /Users/mollyqiao/step2_dataset_outputs


In [57]:
products = [
    {"product": "Scented Candle Set", "price": 18, "stock": 120, "category": "Home Decor"},
    {"product": "Essential Oil Diffuser", "price": 35, "stock": 60, "category": "Wellness"},
    {"product": "Lavender Essential Oil", "price": 15, "stock": 200, "category": "Wellness"},
    {"product": "Aromatherapy Bath Salts", "price": 14, "stock": 90, "category": "Wellness"},
    {"product": "Ceramic Coffee Mug", "price": 10, "stock": 150, "category": "Kitchen"},
    {"product": "Gourmet Coffee Sampler", "price": 22, "stock": 80, "category": "Food"},
    {"product": "Wooden Picture Frame", "price": 12, "stock": 85, "category": "Home Decor"},
    {"product": "Linen Throw Blanket", "price": 45, "stock": 70, "category": "Home Decor"},
    {"product": "Herbal Tea Sampler", "price": 16, "stock": 95, "category": "Food"},
    {"product": "Tea Mug", "price": 11, "stock": 130, "category": "Kitchen"},
    {"product": "Notebook Journal", "price": 13, "stock": 100, "category": "Stationery"},
    {"product": "Decorative Pen Set", "price": 9, "stock": 110, "category": "Stationery"},
]

In [58]:
product_lookup = {p["product"]: p for p in products}

def build_inventory(product_names):
    return [product_lookup[name] for name in product_names]

## Build Typical, Edge, and Adversarial Cases

Next, we create structured examples for three case types:

- **Typical**: normal, realistic bundle scenarios  
- **Edge**: limited or unusual inventory situations  
- **Adversarial**: weakly related products that test system robustness

In [59]:
cases = [
    {
        "case_id": "T01",
        "case_type": "typical",
        "source": "real-style + synthetic",
        "inventory": build_inventory(["Essential Oil Diffuser", "Lavender Essential Oil", "Aromatherapy Bath Salts", "Scented Candle Set"]),
        "scenario": "A wellness-focused inventory with clearly complementary products.",
        "expected_behavior": "Generate a realistic wellness or spa bundle.",
    },
    {
        "case_id": "T02",
        "case_type": "typical",
        "source": "real-style + synthetic",
        "inventory": build_inventory(["Ceramic Coffee Mug", "Gourmet Coffee Sampler", "Linen Throw Blanket", "Scented Candle Set"]),
        "scenario": "A cozy home and coffee scenario.",
        "expected_behavior": "Generate a comfort or coffee-themed bundle.",
    },
    {
        "case_id": "T03",
        "case_type": "typical",
        "source": "synthetic",
        "inventory": build_inventory(["Notebook Journal", "Decorative Pen Set", "Tea Mug"]),
        "scenario": "A desk and productivity setup.",
        "expected_behavior": "Generate a study or work-oriented bundle.",
    },
    {
        "case_id": "T04",
        "case_type": "typical",
        "source": "synthetic",
        "inventory": build_inventory(["Herbal Tea Sampler", "Tea Mug", "Scented Candle Set"]),
        "scenario": "A tea and relaxation product set.",
        "expected_behavior": "Generate a calm evening or tea bundle.",
    },
    {
        "case_id": "T05",
        "case_type": "typical",
        "source": "synthetic",
        "inventory": build_inventory(["Wooden Picture Frame", "Scented Candle Set", "Linen Throw Blanket"]),
        "scenario": "A home decor and gift scenario.",
        "expected_behavior": "Generate a home or gift bundle.",
    },
    {
        "case_id": "T06",
        "case_type": "typical",
        "source": "synthetic",
        "inventory": build_inventory(["Ceramic Coffee Mug", "Gourmet Coffee Sampler", "Tea Mug"]),
        "scenario": "A small beverage-related inventory.",
        "expected_behavior": "Generate a drink-focused bundle.",
    },

    {
        "case_id": "E01",
        "case_type": "edge",
        "source": "synthetic",
        "inventory": build_inventory(["Essential Oil Diffuser", "Lavender Essential Oil"]),
        "scenario": "Very limited wellness inventory with only two products.",
        "expected_behavior": "Generate a minimal but reasonable bundle.",
    },
    {
        "case_id": "E02",
        "case_type": "edge",
        "source": "synthetic",
        "inventory": build_inventory(["Notebook Journal", "Decorative Pen Set"]),
        "scenario": "Only two stationery products are available.",
        "expected_behavior": "Avoid overcomplicating the recommendation.",
    },
    {
        "case_id": "E03",
        "case_type": "edge",
        "source": "synthetic",
        "inventory": build_inventory(["Wooden Picture Frame", "Scented Candle Set"]),
        "scenario": "Only two home decor products are available.",
        "expected_behavior": "Generate a simple decor bundle.",
    },

    {
        "case_id": "A01",
        "case_type": "adversarial",
        "source": "synthetic",
        "inventory": build_inventory(["Wooden Picture Frame", "Gourmet Coffee Sampler", "Lavender Essential Oil"]),
        "scenario": "Products are only weakly related across categories.",
        "expected_behavior": "Be cautious and avoid forcing an unrealistic bundle.",
    },
    {
        "case_id": "A02",
        "case_type": "adversarial",
        "source": "synthetic",
        "inventory": build_inventory(["Decorative Pen Set", "Aromatherapy Bath Salts", "Ceramic Coffee Mug"]),
        "scenario": "Mixed products with unclear customer use case.",
        "expected_behavior": "Find the most plausible theme or remain conservative.",
    },
    {
        "case_id": "A03",
        "case_type": "adversarial",
        "source": "synthetic",
        "inventory": build_inventory(["Tea Mug", "Wooden Picture Frame", "Notebook Journal"]),
        "scenario": "Cross-category products with weak complementarity.",
        "expected_behavior": "Avoid overclaiming bundle coherence.",
    },
]

In [60]:
df = pd.DataFrame(cases)

df["inventory_json"] = df["inventory"].apply(lambda x: json.dumps(x, ensure_ascii=False))

def make_user_prompt(inventory_json):
    return f"""
Using the following inventory, recommend 2 product bundles.

For each bundle include:
- bundle name
- products included
- original total price
- discounted bundle price
- short marketing tagline

Inventory:
{inventory_json}
""".strip()

df["user_prompt"] = df["inventory_json"].apply(make_user_prompt)

df[["case_id", "case_type", "source", "scenario", "expected_behavior"]]

,case_id,case_type,source,scenario,expected_behavior
0,T01,typical,real-style + synthetic,A wellness-focused inventory with clearly comp...,Generate a realistic wellness or spa bundle.
1,T02,typical,real-style + synthetic,A cozy home and coffee scenario.,Generate a comfort or coffee-themed bundle.
2,T03,typical,synthetic,A desk and productivity setup.,Generate a study or work-oriented bundle.
3,T04,typical,synthetic,A tea and relaxation product set.,Generate a calm evening or tea bundle.
4,T05,typical,synthetic,A home decor and gift scenario.,Generate a home or gift bundle.
5,T06,typical,synthetic,A small beverage-related inventory.,Generate a drink-focused bundle.
6,E01,edge,synthetic,Very limited wellness inventory with only two ...,Generate a minimal but reasonable bundle.
7,E02,edge,synthetic,Only two stationery products are available.,Avoid overcomplicating the recommendation.
8,E03,edge,synthetic,Only two home decor products are available.,Generate a simple decor bundle.
9,A01,adversarial,synthetic,Products are only weakly related across catego...,Be cautious and avoid forcing an unrealistic b...


In [61]:
print("Missing values:")
print(df.isna().sum(), "\n")

print("Duplicate case_id count:", df["case_id"].duplicated().sum(), "\n")

print("Case type distribution:")
print(df["case_type"].value_counts())

Missing values:
case_id              0
case_type            0
source               0
inventory            0
scenario             0
expected_behavior    0
inventory_json       0
user_prompt          0
dtype: int64 

Duplicate case_id count: 0 

Case type distribution:
case_type
typical        6
edge           3
adversarial    3
Name: count, dtype: int64


In [62]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
dev_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
dev_df = df.iloc[train_end:dev_end].copy()
test_df = df.iloc[dev_end:].copy()

train_df["split"] = "train"
dev_df["split"] = "dev"
test_df["split"] = "test"

full_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

print("Train size:", len(train_df))
print("Dev size:", len(dev_df))
print("Test size:", len(test_df))

Train size: 8
Dev size: 2
Test size: 2


In [63]:
full_df[["case_id", "case_type", "split"]].sort_values(by="case_id")

,case_id,case_type,split
1,A01,adversarial,train
0,A02,adversarial,train
7,A03,adversarial,train
11,E01,edge,test
9,E02,edge,dev
3,E03,edge,train
2,T01,typical,train
6,T02,typical,train
5,T03,typical,train
10,T04,typical,test


In [64]:
from pathlib import Path

output_dir = Path.home() / "Desktop" / "bundle_dataset"
output_dir.mkdir(exist_ok=True)

In [65]:
full_path = output_dir / "bundle_dataset_full.csv"
train_path = output_dir / "bundle_dataset_train.csv"
dev_path = output_dir / "bundle_dataset_dev.csv"
test_path = output_dir / "bundle_dataset_test.csv"

full_df.to_csv(full_path, index=False, encoding="utf-8")
train_df.to_csv(train_path, index=False, encoding="utf-8")
dev_df.to_csv(dev_path, index=False, encoding="utf-8")
test_df.to_csv(test_path, index=False, encoding="utf-8")

print("Saved:", full_path)
print("Saved:", train_path)
print("Saved:", dev_path)
print("Saved:", test_path)

Saved: /Users/mollyqiao/Desktop/bundle_dataset/bundle_dataset_full.csv
Saved: /Users/mollyqiao/Desktop/bundle_dataset/bundle_dataset_train.csv
Saved: /Users/mollyqiao/Desktop/bundle_dataset/bundle_dataset_dev.csv
Saved: /Users/mollyqiao/Desktop/bundle_dataset/bundle_dataset_test.csv


In [66]:
summary = pd.DataFrame({
    "metric": ["total_cases", "typical_cases", "edge_cases", "adversarial_cases", "train_size", "dev_size", "test_size"],
    "value": [
        len(full_df),
        (full_df["case_type"] == "typical").sum(),
        (full_df["case_type"] == "edge").sum(),
        (full_df["case_type"] == "adversarial").sum(),
        len(train_df),
        len(dev_df),
        len(test_df),
    ]
})

summary

,metric,value
0,total_cases,12
1,typical_cases,6
2,edge_cases,3
3,adversarial_cases,3
4,train_size,8
5,dev_size,2
6,test_size,2


## Step 2 Summary

In this step, we constructed a small structured dataset for evaluating the product bundle recommendation system.

The dataset contains **12 curated cases**, including:

- **6 typical cases** representing normal retail bundle scenarios
- **3 edge cases** representing constrained inventory situations
- **3 adversarial cases** designed to challenge bundle logic and product coherence

Most cases are **synthetic but designed to simulate realistic small retail inventory situations** using a base set of lifestyle and home products.

The dataset was then randomly split into:

- **8 training cases**
- **2 development cases**
- **2 test cases**

This dataset provides a simple but structured foundation for later experiments, including prompt evaluation, fine-tuning, and possible RAG-based grounding.